# Proses ETL


In [61]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings

## Extract

In [62]:
df = pd.read_csv('/hotel_booking.csv')

print("=" * 80)
print("1. EXTRACT - HASIL PEMBACAAN DATASET")
print("=" * 80)

print("\n--- a. head() - 5 Baris Pertama ---")
print(df.head())

print("\n--- b. info() - Informasi Dataset ---")
print(df.info())

print("\n--- c. describe() - Statistik Deskriptif ---")
print(df.describe())

print("\n--- d. Shape Dataset ---")
print(f"Jumlah Baris: {df.shape[0]}")
print(f"Jumlah Kolom: {df.shape[1]}")

1. EXTRACT - HASIL PEMBACAAN DATASET

--- a. head() - 5 Baris Pertama ---
          hotel  is_canceled  lead_time  arrival_date_year arrival_date_month  \
0  Resort Hotel            0        342               2015               July   
1  Resort Hotel            0        737               2015               July   
2  Resort Hotel            0          7               2015               July   
3  Resort Hotel            0         13               2015               July   
4  Resort Hotel            0         14               2015               July   

   arrival_date_week_number  arrival_date_day_of_month  \
0                        27                          1   
1                        27                          1   
2                        27                          1   
3                        27                          1   
4                        27                          1   

   stays_in_weekend_nights  stays_in_week_nights  adults  ...  customer_type  \
0         

In [63]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Data Understanding

### 2.1 Identifikasi Missing Value

In [64]:
# 2.1 Identifikasi Missing Value
print("\n--- a. Missing Value Analysis ---")
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df)) * 100
missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing Percentage (%)': missing_pct
})
missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values('Missing Count', ascending=False)
print(missing_df)


--- a. Missing Value Analysis ---
          Missing Count  Missing Percentage (%)
company          112593               94.306893
agent             16340               13.686238
country             488                0.408744
children              4                0.003350


### 2.2 Identifikasi Tipe Data

In [65]:
# 2.2 Identifikasi Tipe Data
print("\n--- b. Tipe Data per Kolom ---")
dtype_df = pd.DataFrame({
    'Kolom': df.columns,
    'Tipe Data': df.dtypes.values,
    'Non-Null Count': df.count().values
})
print(dtype_df.to_string(index=False))


--- b. Tipe Data per Kolom ---
                         Kolom Tipe Data  Non-Null Count
                         hotel    object          119390
                   is_canceled     int64          119390
                     lead_time     int64          119390
             arrival_date_year     int64          119390
            arrival_date_month    object          119390
      arrival_date_week_number     int64          119390
     arrival_date_day_of_month     int64          119390
       stays_in_weekend_nights     int64          119390
          stays_in_week_nights     int64          119390
                        adults     int64          119390
                      children   float64          119386
                        babies     int64          119390
                          meal    object          119390
                       country    object          118902
                market_segment    object          119390
          distribution_channel    object          119390

### 2.3 Kolom-kolom Penting

In [66]:
# 2.3 Kolom-kolom Penting
print("\n--- c. Penjelasan Kolom-Kolom Penting ---")
kolom_penting = {
    'hotel': 'Jenis hotel (Resort Hotel / City Hotel)',
    'is_canceled': 'Status pembatalan (0 = tidak dibatalkan, 1 = dibatalkan)',
    'lead_time': 'Jumlah hari antara booking dan kedatangan',
    'arrival_date_year/month/week_number/day_of_month': 'Tanggal kedatangan tamu',
    'stays_in_weekend_nights': 'Jumlah malam menginap di akhir pekan',
    'stays_in_week_nights': 'Jumlah malam menginap di hari kerja',
    'adults/children/babies': 'Jumlah tamu dewasa, anak, dan bayi',
    'meal': 'Tipe paket makan (BB, HB, FB, SC)',
    'country': 'Negara asal tamu',
    'market_segment': 'Segmen pasar (Online TA, Direct, Corporate, dll)',
    'distribution_channel': 'Kanal distribusi booking',
    'adr': 'Average Daily Rate - tarif rata-rata per malam',
    'required_car_parking_spaces': 'Jumlah tempat parkir yang dibutuhkan',
    'total_of_special_requests': 'Jumlah permintaan khusus tamu',
    'reservation_status': 'Status reservasi (Check-Out, Canceled, No-Show)',
    'reservation_status_date': 'Tanggal status reservasi terakhir',
    'customer_type': 'Tipe pelanggan (Transient, Contract, Group, dll)'
}
for k, v in kolom_penting.items():
    print(f"  • {k}: {v}")


--- c. Penjelasan Kolom-Kolom Penting ---
  • hotel: Jenis hotel (Resort Hotel / City Hotel)
  • is_canceled: Status pembatalan (0 = tidak dibatalkan, 1 = dibatalkan)
  • lead_time: Jumlah hari antara booking dan kedatangan
  • arrival_date_year/month/week_number/day_of_month: Tanggal kedatangan tamu
  • stays_in_weekend_nights: Jumlah malam menginap di akhir pekan
  • stays_in_week_nights: Jumlah malam menginap di hari kerja
  • adults/children/babies: Jumlah tamu dewasa, anak, dan bayi
  • meal: Tipe paket makan (BB, HB, FB, SC)
  • country: Negara asal tamu
  • market_segment: Segmen pasar (Online TA, Direct, Corporate, dll)
  • distribution_channel: Kanal distribusi booking
  • adr: Average Daily Rate - tarif rata-rata per malam
  • required_car_parking_spaces: Jumlah tempat parkir yang dibutuhkan
  • total_of_special_requests: Jumlah permintaan khusus tamu
  • reservation_status: Status reservasi (Check-Out, Canceled, No-Show)
  • reservation_status_date: Tanggal status reservasi

### 2.4 Potensi Masalah Data

In [67]:
# 2.4 Potensi Masalah Data
print("\n--- d. Potensi Masalah Data ---")
print("1. Missing Value:")
print(f"   - children: {df['children'].isnull().sum()} missing ({(df['children'].isnull().sum()/len(df)*100):.2f}%)")
print(f"   - country: {df['country'].isnull().sum()} missing ({(df['country'].isnull().sum()/len(df)*100):.2f}%)")
print(f"   - agent: {df['agent'].isnull().sum()} missing ({(df['agent'].isnull().sum()/len(df)*100):.2f}%)")
print(f"   - company: {df['company'].isnull().sum()} missing ({(df['company'].isnull().sum()/len(df)*100):.2f}%)")

print("\n2. Data Tidak Valid:")
print(f"   - adr < 0: {(df['adr'] < 0).sum()} baris")
print(f"   - adr = 0: {(df['adr'] == 0).sum()} baris")
print(f"   - adults = 0: {(df['adults'] == 0).sum()} baris")
print(f"   - total_nights = 0 (weekend + weekday = 0): {((df['stays_in_weekend_nights'] + df['stays_in_week_nights']) == 0).sum()} baris")

print("\n3. Tipe Data yang Perlu Diubah:")
print("   - reservation_status_date: object → datetime")
print("   - children: float64 → int64")
print("   - agent, company: float64 → object/string (kode ID)")

print("\n4. Data Duplikat:")
duplicates = df.duplicated().sum()
print(f"   - Jumlah baris duplikat: {duplicates}")


--- d. Potensi Masalah Data ---
1. Missing Value:
   - children: 4 missing (0.00%)
   - country: 488 missing (0.41%)
   - agent: 16340 missing (13.69%)
   - company: 112593 missing (94.31%)

2. Data Tidak Valid:
   - adr < 0: 1 baris
   - adr = 0: 1959 baris
   - adults = 0: 403 baris
   - total_nights = 0 (weekend + weekday = 0): 715 baris

3. Tipe Data yang Perlu Diubah:
   - reservation_status_date: object → datetime
   - children: float64 → int64
   - agent, company: float64 → object/string (kode ID)

4. Data Duplikat:
   - Jumlah baris duplikat: 0


## Transform

### Membuat salinan untuk transformasi

In [68]:
# 3. TRANSFORM - Membuat salinan untuk transformasi
df_clean = df.copy()

### Data Cleaning

#### Missing Value

In [69]:
# children: 4 missing → isi dengan 0 (asumsi tidak ada anak)
print(f"\n  • children: {df_clean['children'].isnull().sum()} missing → diisi dengan 0")
print("    Alasan: Jumlah anak yang kosong diasumsikan 0 (tidak membawa anak)")
df_clean['children'] = df_clean['children'].fillna(0)


  • children: 4 missing → diisi dengan 0
    Alasan: Jumlah anak yang kosong diasumsikan 0 (tidak membawa anak)


In [70]:
# country: 488 missing → isi dengan 'Unknown'
print(f"\n  • country: {df_clean['country'].isnull().sum()} missing → diisi dengan 'Unknown'")
print("    Alasan: Negara tidak diketahui, tidak boleh dihapus karena data lain valid")
df_clean['country'] = df_clean['country'].fillna('Unknown')


  • country: 488 missing → diisi dengan 'Unknown'
    Alasan: Negara tidak diketahui, tidak boleh dihapus karena data lain valid


In [71]:
# agent: 16340 missing → isi dengan 'No Agent'
print(f"\n  • agent: {df_clean['agent'].isnull().sum()} missing → diisi dengan 'No Agent'")
print("    Alasan: Booking tanpa agent (Direct booking) adalah kategori valid")
df_clean['agent'] = df_clean['agent'].fillna('No Agent')


  • agent: 16340 missing → diisi dengan 'No Agent'
    Alasan: Booking tanpa agent (Direct booking) adalah kategori valid


In [72]:
# company: 112593 missing (94.3%) → Drop kolom karena terlalu banyak missing
print(f"\n  • company: {df_clean['company'].isnull().sum()} missing ({(df_clean['company'].isnull().sum()/len(df_clean)*100):.1f}%)")
print("    Alasan: 94.3% data kosong, kolom ini tidak informatif untuk analisis BI")
print("    Keputusan: DROP kolom 'company'")
df_clean = df_clean.drop('company', axis=1)


  • company: 112593 missing (94.3%)
    Alasan: 94.3% data kosong, kolom ini tidak informatif untuk analisis BI
    Keputusan: DROP kolom 'company'


In [73]:
print(f"\n Setelah pembersihan: Missing value tersisa = {df_clean.isnull().sum().sum()}")


 Setelah pembersihan: Missing value tersisa = 0


#### Menghapus Data Tidak Valid

ALASAN DATA INVALID HARUS DIHAPUS:

1. KUALITAS DATA (DATA QUALITY): Data warehouse memerlukan data yang
   representatif dan akurat. Data invalid adalah noise yang merusak analisis.

2. AKURASI KPI: ADR (Average Daily Rate) <= 0 berarti hotel tidak mendapat
   revenue. Jika disertakan, KPI "Total Revenue" dan "RevPAR" akan salah.

3. ANALISIS PREDIKTIF: Model machine learning untuk prediksi cancellation
   atau demand forecasting akan terpengaruh negatif oleh outlier invalid.

4. REPORTING FINANSIAL: Laporan keuangan untuk stakeholder tidak boleh
   mencakup transaksi dengan nilai negatif atau nol yang tidak masuk akal.

5. DDS INTEGRITY: Fact table dalam DDS menyimpan measures (fact) yang harus
   valid. Invalid measures akan merusak rollup dan drill-down operations.

In [74]:
# Cek data invalid sebelum penghapusan
invalid_adr_neg = (df_clean['adr'] < 0).sum()
invalid_adr_zero = (df_clean['adr'] == 0).sum()
invalid_adults_zero = (df_clean['adults'] == 0).sum()
invalid_nights_zero = ((df_clean['stays_in_weekend_nights'] + df_clean['stays_in_week_nights']) == 0).sum()

print(f"\n  • adr < 0 (negative): {invalid_adr_neg} baris")
print(f"    Contoh: adr = -{df_clean[df_clean['adr'] < 0]['adr'].abs().max() if invalid_adr_neg > 0 else 'N/A'}")
print(f"    Dampak: Revenue negatif → Total Revenue salah → Laporan keuangan invalid")

print(f"\n  • adr = 0 (zero): {invalid_adr_zero} baris")
print(f"    Dampak: Revenue nol untuk booking yang valid → RevPAR = 0 →")
print(f"           Analisis profitabilitas hotel menjadi tidak akurat")

print(f"\n  • adults = 0: {invalid_adults_zero} baris")
print(f"    Dampak: Booking tanpa tamu dewasa adalah data anomali →")
print(f"           Occupancy rate dan ADR per tamu menjadi tidak valid")

print(f"\n  • total_nights = 0: {invalid_nights_zero} baris")
print(f"    Dampak: Booking tanpa menginap → Length of Stay (LOS) = 0 →")
print(f"           Analisis durasi menginap dan revenue per night gagal")


  • adr < 0 (negative): 1 baris
    Contoh: adr = -6.38
    Dampak: Revenue negatif → Total Revenue salah → Laporan keuangan invalid

  • adr = 0 (zero): 1959 baris
    Dampak: Revenue nol untuk booking yang valid → RevPAR = 0 →
           Analisis profitabilitas hotel menjadi tidak akurat

  • adults = 0: 403 baris
    Dampak: Booking tanpa tamu dewasa adalah data anomali →
           Occupancy rate dan ADR per tamu menjadi tidak valid

  • total_nights = 0: 715 baris
    Dampak: Booking tanpa menginap → Length of Stay (LOS) = 0 →
           Analisis durasi menginap dan revenue per night gagal


In [75]:
# Hapus adr < 0
print(f"\n  Step 1: Hapus adr < 0 → {invalid_adr_neg} baris dihapus")
df_clean = df_clean[df_clean['adr'] >= 0]


  Step 1: Hapus adr < 0 → 1 baris dihapus


In [76]:
# Hapus adults = 0
print(f"  Step 2: Hapus adults = 0 → {invalid_adults_zero} baris dihapus")
df_clean = df_clean[df_clean['adults'] > 0]

  Step 2: Hapus adults = 0 → 403 baris dihapus


In [77]:
# Hapus total nights = 0 (kecuali untuk canceled bookings yang mungkin valid)
# Namun untuk DDS, kita hanya simpan data dengan nights > 0 untuk analisis menginap
print(f"  Step 3: Hapus total_nights = 0 → {invalid_nights_zero} baris dihapus")
df_clean = df_clean[(df_clean['stays_in_weekend_nights'] + df_clean['stays_in_week_nights']) > 0]


  Step 3: Hapus total_nights = 0 → 715 baris dihapus


In [78]:
print(f"\n  Jumlah baris setelah penghapusan: {len(df_clean)}")
print(f"  Total baris dihapus: {len(df) - len(df_clean)}")
print(f"  Persentase data yang dipertahankan: {(len(df_clean)/len(df)*100):.2f}%")


  Jumlah baris setelah penghapusan: 118341
  Total baris dihapus: 1049
  Persentase data yang dipertahankan: 99.12%


#### Menghapus Data Duplikat

In [79]:
# Cek duplikat berdasarkan semua kolom
duplicates_all = df_clean.duplicated().sum()
print(f"\n  • Duplikat exact (semua kolom sama): {duplicates_all} baris")


  • Duplikat exact (semua kolom sama): 0 baris


### Data Transformation

##### Rename semua kolom ke lowercase

In [80]:
print("\nSebelum:")
print(f"   Kolom sample: {list(df_clean.columns[:5])}")

# Rename semua kolom ke lowercase
df_clean.columns = df_clean.columns.str.lower()

print("\nSesudah:")
print(f"   Kolom sample: {list(df_clean.columns[:5])}")
print(f"\nTotal kolom setelah rename: {len(df_clean.columns)}")


Sebelum:
   Kolom sample: ['hotel', 'is_canceled', 'lead_time', 'arrival_date_year', 'arrival_date_month']

Sesudah:
   Kolom sample: ['hotel', 'is_canceled', 'lead_time', 'arrival_date_year', 'arrival_date_month']

Total kolom setelah rename: 35


##### Mengubah Tipe Data

ALASAN TIPE DATA HARUS SESUAI DALAM DDS:

1. OPTIMALISASI STORAGE: Tipe data yang tepat mengurangi ukuran storage.
   Contoh: int32 vs float64 untuk ID dapat menghemat 50% storage.

2. KECEPATAN QUERY: Database dapat melakukan index dan partition lebih efisien
   pada tipe data yang sesuai. Query pada datetime jauh lebih cepat daripada string.

3. AKURASI AGREGASI: Tipe data numerik memastikan SUM, AVG, MIN, MAX berjalan
   dengan benar. String yang menyimpan angka tidak dapat diagregasi.

4. FILTER & SORTING: Filter tanggal (BETWEEN, >, <) hanya berfungsi pada datetime.
   String comparison pada tanggal menghasilkan hasil yang salah.

5. DASHBOARD BI: Tools BI mengenali tipe data untuk menentukan visualisasi
   yang tepat. Datetime → time series chart, numeric → bar/line chart.

6. FOREIGN KEY INTEGRITY: Dalam DDS, foreign keys harus memiliki tipe data
   yang sama dengan primary key di dimension table untuk join yang valid.

In [81]:
print("\nROSES PERUBAHAN TIPE DATA:")
print("\nSebelum perubahan:")
print(df_clean[['children', 'agent', 'reservation_status_date']].dtypes)



ROSES PERUBAHAN TIPE DATA:

Sebelum perubahan:
children                   float64
agent                       object
reservation_status_date     object
dtype: object


In [82]:
# children: float64 → int64 (jumlah anak harus integer)
print("\n  • children: float64 → int64")
print("    Alasan: Jumlah anak adalah bilangan bulat, tidak ada desimal")
df_clean['children'] = df_clean['children'].astype(int)



  • children: float64 → int64
    Alasan: Jumlah anak adalah bilangan bulat, tidak ada desimal


In [83]:
# agent: float64 → object/string (kode agent, bukan angka untuk kalkulasi)
print("\n  • agent: float64 → object (string)")
print("    Alasan: 'agent' adalah kode ID, bukan nilai numerik untuk operasi matematika")
df_clean['agent'] = df_clean['agent'].astype(str)


  • agent: float64 → object (string)
    Alasan: 'agent' adalah kode ID, bukan nilai numerik untuk operasi matematika


In [84]:
# reservation_status_date: object → datetime
print("\n  • reservation_status_date: object → datetime64")
print("    Alasan: Tanggal memerlukan tipe datetime untuk filter, sorting,")
print("           time-series analysis, dan perhitungan durasi")
df_clean['reservation_status_date'] = pd.to_datetime(df_clean['reservation_status_date'])


  • reservation_status_date: object → datetime64
    Alasan: Tanggal memerlukan tipe datetime untuk filter, sorting,
           time-series analysis, dan perhitungan durasi


In [85]:
print("\nSesudah perubahan:")
print(df_clean[['children', 'agent', 'reservation_status_date']].dtypes)


Sesudah perubahan:
children                            int64
agent                              object
reservation_status_date    datetime64[ns]
dtype: object


### Feature Engineering

KENAPA FEATURE ENGINEERING PENTING DALAM DDS:

1. PRE-AGGREGATED MEASURES: DDS (Dimensional Data Store) menyimpan data
   dalam bentuk yang siap untuk analisis. Feature engineering membuat
   measures yang sering digunakan dalam query BI menjadi kolom tersendiri.

2. QUERY PERFORMANCE: Kolom yang sering diagregasi (SUM, AVG) sebaiknya
   di-precompute untuk mengurangi waktu query pada data warehouse.

3. BUSINESS SEMANTICS: Kolom seperti 'revenue' dan 'total_nights' memiliki
   makna bisnis yang jelas, memudahkan business user memahami data.

4. STAR SCHEMA DESIGN: Fact table dalam DDS berisi measures (fact) yang
   berasal dari feature engineering. Dimension table berisi attributes.

5. SELF-SERVICE BI: Business user dapat membuat dashboard tanpa menulis
   query kompleks karena measures sudah tersedia sebagai kolom.

## Load

In [92]:
powerbi_df = df_clean[[
    'hotel',
    'is_canceled',
    'lead_time',
    'arrival_date_year',
    'arrival_date_month',
    'arrival_date_week_number',
    'arrival_date_day_of_month',
    'stays_in_weekend_nights',
    'stays_in_week_nights',
    'adults',
    'children',
    'babies',
    'meal',
    'country',
    'market_segment',
    'distribution_channel',
    'reserved_room_type',
    'assigned_room_type',
    'booking_changes',
    'deposit_type',
    'customer_type',
    'adr',
    'required_car_parking_spaces',
    'total_of_special_requests',
    'total_nights',
    'revenue'
]].copy()

powerbi_df.to_csv(
    'powerbi_hotel.csv',
    index=False
)

### Pembuatan Fact Table

In [93]:
from google.colab import files

files.download('powerbi_hotel.csv')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>